<a href="https://colab.research.google.com/github/SoupBoi1/TLS-Fingerprinting-Malicious-Detection/blob/main/TLSMaliciousDetection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
#TLSFingerprintingMaliciousDetectionNotebook1

#author Matt Lutjen, Brady Bangasser, Sudipta Halder,

#Importing data and libs

In [4]:
!pip install optuna
!pip install optuna-dashboard
!pip install lime
!pip install imbalanced-learn
!pip install xgboost


In [5]:
import optuna

/Users/sudiptahalder/Documents/school/TLS-Fingerprinting-Malicious-Detection/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
import numpy as np
import json
import pandas as pd
import matplotlib.pyplot as plt
import requests
import os, random

In [7]:


#preprocessing
from sklearn.preprocessing import normalize
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import TargetEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import LabelBinarizer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression


#imbalance
from sklearn.impute import SimpleImputer
from imblearn.datasets import make_imbalance
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import TomekLinks


#datasets
from sklearn.datasets import load_iris
from imblearn.datasets import make_imbalance
from sklearn.datasets import make_classification
from sklearn.base import BaseEstimator, ClassifierMixin, RegressorMixin

from sklearn.model_selection import train_test_split


#imbalanced learning
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import TomekLinks


#models
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.linear_model import LinearRegression
from sklearn import tree
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import OneClassSVM
from sklearn.neural_network import MLPClassifier
from sklearn.svm import OneClassSVM


#matrics
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.metrics import roc_auc_score
from sklearn.metrics import precision_recall_curve, auc
from sklearn.metrics import average_precision_score
from sklearn.metrics import adjusted_rand_score,normalized_mutual_info_score
from sklearn.metrics import silhouette_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_curve, auc, ConfusionMatrixDisplay

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.model_selection import cross_val_score


#interpretability
import lime
import lime.lime_tabular

import seaborn as sns


In [8]:
import tensorflow as tf
from tensorflow import keras

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import BatchNormalization,Dense, Conv2D, Flatten, Reshape
from tensorflow.keras.layers import Activation
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import Input
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import BatchNormalization

In [9]:
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))


Num GPUs Available:  1


In [10]:
seed = 2
random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(
    seed
)
# Recommended for modern NumPy
# rng = np.random.default_rng(seed)
os.environ['PYTHONHASHSEED'] = str(seed)

In [11]:
df = pd.read_csv('https://raw.githubusercontent.com/SoupBoi1/TLS-Fingerprinting-Malicious-Detection/refs/heads/main/dataset/dataset.csv')#getting the data

In [12]:

display(df.head())

,SrcPort,SNI,AppName,DstPort,JA3Shash,JA3hash,JA4hash,Type,SrcIP,Version,Filename,OrgName,DstIP,JA4Shash,Label,Attack
0,58688,officehymy.com,sodinokibi,443,61be9ce3d068c08ff99a857f62352f9d,a0e9f5d64349fb13191bc781f81f42e1,t12d190800_d83cc789557e_7af1ed941c26,M,10.127.0.109,0.0,tests/malware2/SODINOKIBI/240602-km899agd6t.be...,Japan Network Information Center; XSERVER20 (JP),157.112.183.48,t120400_c02f_460f64128655,SODINOKIBI,1
1,51187,sf16-muse-va.ibytedtos.com,star_quiz_01,443,42ec7b1db61428bf1cc6e01b9ef02b04,6ec2896feff5746955f700c0023f5804,t12d1409h1_c866b44c5a26_b39be8c56a14,M,192.168.0.100,0.0,tests/malware4/android11/star_quiz_01-extracte...,Slovak Telecom Network Administrator; BA-WEBHO...,212.5.219.10,t1206h1_c02c_e1dda4771ae8,MALWARE,1
2,61625,avatars.mds.yandex.net,zloader,443,4ee87de303b9c8138441e6527cbeea3e,cd08e31494f9531f560d64c695473da9,t13d1516h2_8daaf6152771_e5627efa2ab1,M,10.127.0.108,0.0,tests/malware2/ZLOADER/240525-adwbxsga76.behav...,YANDEX LLC; YANDEX-87-250-247 (RU),87.250.247.182,t1304h2_1302_a56c5b993250,ZLOADER,1
3,50972,js.stripe.com,asyncrat,443,60c22edca21e0598ad28f8d78c8dedcd,579ccef312d18482fc42e2b822ca2430,t13d1715h2_5b57614c22b0_a815a0c236aa,M,10.127.0.20,0.0,tests/malware2/ASYNCRAT/240608-e15lbsgh9x.beha...,Amazon.com; Inc.; AMAZON-CF,18.245.86.52,t1304h2_1301_a56c5b993250,ASYNCRAT,1
4,50579,rps-svcs.oracle.com,bazarbackdoor,443,a860078aa51e1102b117d1f8a1437b72,2a458dd9c65afbcf591cd8c2a194b804,t12d210600_b973bfd88a0e_1da50ec048a3,M,10.127.0.3,0.0,tests/malware2/BAZARBACKDOOR/230429-ytv8vsbh95...,Akamai International; BV; AIBV,23.222.50.60,t120400_c014_cbb8871a0652,BAZARBACKDOOR,1


In [13]:
df.info()#info
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30664 entries, 0 to 30663
Data columns (total 16 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   SrcPort   30664 non-null  int64  
 1   SNI       30239 non-null  object 
 2   AppName   30664 non-null  object 
 3   DstPort   30664 non-null  int64  
 4   JA3Shash  28951 non-null  object 
 5   JA3hash   30664 non-null  object 
 6   JA4hash   30664 non-null  object 
 7   Type      30414 non-null  object 
 8   SrcIP     30664 non-null  object 
 9   Version   28951 non-null  float64
 10  Filename  28951 non-null  object 
 11  OrgName   30664 non-null  object 
 12  DstIP     30664 non-null  object 
 13  JA4Shash  28951 non-null  object 
 14  Label     29149 non-null  object 
 15  Attack    30664 non-null  int64  
dtypes: float64(1), int64(3), object(12)
memory usage: 3.7+ MB


,SrcPort,DstPort,Version,Attack
count,30664.000000,30664.000000,28951.0,30664.000000
mean,55442.169482,583.606933,0.0,0.787666
std,4529.980500,1485.352244,0.0,0.408966
min,33139.000000,80.000000,0.0,0.000000
25%,51291.750000,443.000000,0.0,1.000000
50%,54972.500000,443.000000,0.0,1.000000
75%,58615.250000,443.000000,0.0,1.000000
max,65533.000000,49816.000000,0.0,1.000000


In [14]:
df["DstIP"].unique()

array(['157.112.183.48', '212.5.219.10', '87.250.247.182', ...,
       '52.26.253.153', '2.22.89.33', '66.70.219.208'], dtype=object)

#Pre-possessing and Data Analysis

encoding and spliting the IPs

In [15]:
SrcIP_1=[]
SrcIP_2=[]
SrcIP_3=[]
SrcIP_4=[]
for i in df["SrcIP"]:

  #print(i.split('.'))
  SrcIP_1.append(int(i.split('.')[0]))
  SrcIP_2.append(int(i.split('.')[1]))
  SrcIP_3.append(int(i.split('.')[2]))
  SrcIP_4.append(int(i.split('.')[3]))
print(np.shape(SrcIP_1))
print(np.shape(SrcIP_2))
print(np.shape(SrcIP_3))
print(np.shape(SrcIP_4))

(30664,)
(30664,)
(30664,)
(30664,)


In [16]:
DstIP_1=[]
DstIP_2=[]
DstIP_3=[]
DstIP_4=[]
for i in df["DstIP"]:

  #print(i.split('.'))
  DstIP_1.append(int(i.split('.')[0]))
  DstIP_2.append(int(i.split('.')[1]))
  DstIP_3.append(int(i.split('.')[2]))
  DstIP_4.append(int(i.split('.')[3]))

jA4

In [17]:
def JA4_parameters_extration(JA4_A):
  protocal,TLS_version,SNI,Nciper,Nextension,ALPN=None,None,None,None,None,None
  if(len(JA4_A)==10):
    protocal=JA4_A[0]
    TLS_version=(JA4_A[1:3])
    SNI=JA4_A[3]
    Nciper=int(JA4_A[4:6])
    Nextension=int(JA4_A[6:7])
    ALPN=JA4_A[8:]
  return [protocal,TLS_version,SNI,Nciper,Nextension,ALPN]

def JA4S_parameters_extration(JA4S_A):
  protocal,TLS_version,Nextension,ALPN=None,None,None,None
  if(len(JA4S_A)==7):
    protocal=JA4S_A[0]
    TLS_version=(JA4S_A[1:3])
    Nextension=int(JA4S_A[3:5])
    ALPN=JA4S_A[5:]
  return [protocal,TLS_version,Nextension,ALPN]

def JA4H_parameters_extration(JA4H_A):
  method,version,cookie,ref,NHeaders,AL=None,None,None,None,None,None
  if(len(JA4H_A)==12):
    method=JA4H_A[0:2]
    version=(JA4H_A[2:4])
    cookie=JA4H_A[4]
    ref=JA4H_A[5]
    NHeaders=JA4H_A[6:8]
    AL=JA4H_A[8:]
  return [method,version,cookie,ref,NHeaders,AL]

print(np.array(JA4_parameters_extration('t13d1516h2')))
print(JA4S_parameters_extration('t120400'))
print(JA4H_parameters_extration('ge20cr13enus'))

['t' '13' 'd' '15' '1' 'h2']
['t', '12', 4, '00']
['ge', '20', 'c', 'r', '13', 'enus']


In [18]:
JA4A=[]
JA4B=[]
JA4C=[]
Protocal =[]
TLS_Version =[]
Nextension=[]
SNI= []
Nciper=[]
ALPN =[]

for i in df["JA4hash"]:
  array = JA4_parameters_extration(i.split('_')[0])
  Protocal.append(array[0])
  TLS_Version.append(array[1])
  SNI.append(array[2])
  Nciper.append(array[3])
  Nextension.append(array[4])
  ALPN.append(array[5])
  #print(i.split('.'))
  JA4A.append(i.split('_')[0])
  JA4B.append(i.split('_')[1])
  JA4C.append(i.split('_')[2])


In [19]:
JA4SA=[]
JA4SB=[]
JA4SC=[]
ProtocalS =[]
TLS_VersionS =[]
NextensionS=[]
ALPNS =[]

for i in df["JA4Shash"]:

  if i is not None and type(i)==str:
      array = JA4S_parameters_extration(i.split('_')[0])
      ProtocalS.append(array[0])
      TLS_VersionS.append(array[1])
      NextensionS.append(array[2])
      ALPNS.append(array[3])

      JA4SA.append(i.split('_')[0])
      JA4SB.append(i.split('_')[1])
      JA4SC.append(i.split('_')[2])
  else:
    #print(i)
    ProtocalS.append(None)
    TLS_VersionS.append(None)
    NextensionS.append(None)
    ALPNS.append(None)
    JA4SA.append(None)
    JA4SB.append(None)
    JA4SC.append(None)


## 60/20/20 split


In [20]:


features_df = pd.DataFrame({

    'SrcIP_1': SrcIP_1, 'SrcIP_2': SrcIP_2, 'SrcIP_3': SrcIP_3, 'SrcIP_4': SrcIP_4,

    'DstIP_1': DstIP_1, 'DstIP_2': DstIP_2, 'DstIP_3': DstIP_3, 'DstIP_4': DstIP_4,

    'JA4_protocal':Protocal, 'JA4_TLS_version':TLS_Version,'JA4_SNI':SNI,'JA4_Nciper':Nciper ,'JA4_Nextension':Nextension , 'JA4_ALPN':ALPN, 'JA4B': JA4B, 'JA4C': JA4C,

     'JA4S_Protocal': ProtocalS, 'JA4S_TLS_Version': TLS_VersionS, 'JA4S_Nextension': NextensionS, 'JA4S_ALPNS': ALPNS, 'JA4SB': JA4SB, 'JA4SC': JA4SC,

    'SrcPort': df['SrcPort'],

    'DstPort': df['DstPort']#,'Attack': df['Attack']

})

In [21]:
hotencode_labels = ['JA4_TLS_version', 'JA4S_TLS_Version', 'JA4B', 'JA4C', 'JA4SB', 'JA4SC']

In [22]:

X_tempR, X_testR, y_tempR, y_test = train_test_split(features_df, df['Attack'].values, test_size=0.20, random_state=42, stratify=df['Attack'].values)

X_trainR, X_valR, y_train, y_val = train_test_split(X_tempR, y_tempR, test_size=0.25, random_state=42, stratify=y_tempR)

print(f"Training set size: {X_trainR.shape[0]}")

print(f"Validation set size: {X_valR.shape[0]}")

print(f"Testing set size: {X_testR.shape[0]}")

Training set size: 18398
Validation set size: 6133
Testing set size: 6133


In [23]:
X_trainR.columns

Index(['SrcIP_1', 'SrcIP_2', 'SrcIP_3', 'SrcIP_4', 'DstIP_1', 'DstIP_2',
       'DstIP_3', 'DstIP_4', 'JA4_protocal', 'JA4_TLS_version', 'JA4_SNI',
       'JA4_Nciper', 'JA4_Nextension', 'JA4_ALPN', 'JA4B', 'JA4C',
       'JA4S_Protocal', 'JA4S_TLS_Version', 'JA4S_Nextension', 'JA4S_ALPNS',
       'JA4SB', 'JA4SC', 'SrcPort', 'DstPort'],
      dtype='object')

In [24]:
X_trainR.select_dtypes(include=[np.number]).head()


,SrcIP_1,SrcIP_2,SrcIP_3,SrcIP_4,DstIP_1,DstIP_2,DstIP_3,DstIP_4,JA4_Nciper,JA4_Nextension,JA4S_Nextension,SrcPort,DstPort
11659,10,127,0,238,104,19,222,79,12.0,0.0,NaN,49538,443
24143,10,127,0,65,185,199,108,154,15.0,1.0,2.0,49858,443
4509,10,127,0,253,78,40,8,65,12.0,0.0,NaN,50276,443
12177,10,127,0,120,104,21,235,132,12.0,0.0,3.0,49745,443
22573,192,168,3,28,142,251,37,99,17.0,1.0,2.0,59477,443


In [25]:
X_trainR.loc[:, ~X_trainR.columns.isin(hotencode_labels)].select_dtypes(exclude=[np.number]).head()

,JA4_protocal,JA4_SNI,JA4_ALPN,JA4S_Protocal,JA4S_ALPNS
11659,t,d,00,None,None
24143,t,d,h2,t,00
4509,t,d,00,None,None
12177,t,d,00,t,00
22573,t,d,h1,t,00


In [26]:
X_trainR.loc[:, X_trainR.columns.isin(hotencode_labels)].head()


,JA4_TLS_version,JA4B,JA4C,JA4S_TLS_Version,JA4SB,JA4SC
11659,10,d94e65cdb899,5f12c91e28fe,None,None,None
24143,13,8daaf6152771,e5627efa2ab1,13,1301,a56c5b993250
4509,10,d94e65cdb899,5f12c91e28fe,None,None,None
12177,10,d94e65cdb899,5f12c91e28fe,10,c009,344b4dce5a52
22573,13,5b57614c22b0,eca864cca44a,13,1301,234ea6891581


In [27]:
label_trainR = X_trainR.loc[:, ~X_trainR.columns.isin(hotencode_labels)].select_dtypes(exclude=[np.number])
label_testR = X_testR.loc[:, ~X_testR.columns.isin(hotencode_labels)].select_dtypes(exclude=[np.number])
label_valR = X_valR.loc[:, ~X_valR.columns.isin(hotencode_labels)].select_dtypes(exclude=[np.number])

In [28]:
hotencoding_trainR= X_trainR.loc[:, X_trainR.columns.isin(hotencode_labels)]
hotencoding_testR = X_testR.loc[:, X_testR.columns.isin(hotencode_labels)]
hotencoding_valR = X_valR.loc[:, X_valR.columns.isin(hotencode_labels)]

### encoding


#### numeric

In [29]:

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])


numeric_pipeline.fit(np.zeros([10,10]))


,steps,"[('imputer', ...), ('scaler', ...)]"
,transform_input,None
,memory,None
,verbose,False
,missing_values,nan
,strategy,'median'
,fill_value,None
,copy,True
,add_indicator,False
,keep_empty_features,False
,copy,True


In [30]:
X_train_num_t = X_trainR.select_dtypes(include=[np.number])
numeric_pipeline.fit(X_train_num_t)
X_train_num = numeric_pipeline.transform(X_train_num_t)

X_val_num_t = X_valR.select_dtypes(include=[np.number])
#numeric_pipeline.fit(X_val_num_t)
X_val_num = numeric_pipeline.transform(X_val_num_t)

X_test_num_t = X_testR.select_dtypes(include=[np.number])
#numeric_pipeline.fit(X_test_num_t)
X_test_num = numeric_pipeline.transform(X_test_num_t)


#### *categorical*

In [31]:
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

In [32]:
categorical_pipeline.fit(hotencoding_trainR)
X_train_cat = categorical_pipeline.transform(hotencoding_trainR)


X_val_cat = categorical_pipeline.transform(hotencoding_testR)

X_test_cat = categorical_pipeline.transform(hotencoding_valR)

#### Label based

In [33]:
label_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder())
])

In [34]:
label_pipeline.fit(label_trainR)
X_train_label = label_pipeline.transform(label_trainR)


X_val_label = label_pipeline.transform(label_testR)

X_test_label = label_pipeline.transform(label_valR)


In [35]:
X_train = np.concatenate((X_train_num, X_train_cat.toarray(),X_train_label), axis=1)
X_val = np.concatenate((X_val_num, X_val_cat.toarray(),X_val_label), axis=1)
X_test = np.concatenate((X_test_num, X_test_cat.toarray(),X_test_label), axis=1)


### Data Balancing with SMOTE
SMOTE is applied **only** to the training split to prevent data leakage into validation/test sets.

In [36]:
# ── SMOTE: Balance the training set (applied ONLY to training data) ──
print("Class distribution before SMOTE:", dict(zip(*np.unique(y_train, return_counts=True))))

smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print("Class distribution after SMOTE: ", dict(zip(*np.unique(y_train_resampled, return_counts=True))))
print(f"X_train shape before SMOTE: {X_train.shape} → after: {X_train_resampled.shape}")


Class distribution before SMOTE: {0: 3907, 1: 14491}
Class distribution after SMOTE:  {0: 14491, 1: 14491}
X_train shape before SMOTE: (18398, 265) → after: (28982, 265)


# Model Cration, Training, Validation,Testing

In [88]:
import joblib
import os
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.ensemble import IsolationForest

In [113]:
print("--- STARTING MANUAL LOOP TUNING ---")

best_models = {}
predictions = {}


--- STARTING MANUAL LOOP TUNING ---


In [114]:
# Get the current working directory
root_dir = "saved_models"

for item in os.listdir(root_dir):
    item_path = os.path.join(root_dir, item)
    
    if os.path.isdir(item_path):
        if(item =="best_model"):
            for model in os.listdir(item_path):
                model_path = os.path.join(item_path, model)
                if(model.endswith('.joblib')):
                    loaded_model = joblib.load(model_path)
                    best_models[model] =(loaded_model)  
                    print(model)
                elif (model.endswith('.keras')):
                    loaded_model = tf.keras.models.load_model(model_path)
                    best_models[model]=(loaded_model)  

                    print(model)

decision_tree_min_samples_split_2_depth_30.joblib
MLP_hidden_layer_sizes_(50,)_activation_func_relu.joblib
cnn_f59_d44.keras
rf_trees_n_estimators_58_max_depth_10.joblib


In [115]:
# Get the current working directory
root_dir = "savedSMOTE_models"

for item in os.listdir(root_dir):
    item_path = os.path.join(root_dir, item)
    
    if os.path.isdir(item_path):
        if(item =="best_model"):
            for model in os.listdir(item_path):
                model_path = os.path.join(item_path, model)
                if(model.endswith('.joblib')):
                    loaded_model = joblib.load(model_path)
                    best_models["SMOTE "+model] =(loaded_model)  
                    print("SMOTE "+model)
                elif (model.endswith('.keras')):
                    loaded_model = tf.keras.models.load_model(model_path)
                    best_models["SMOTE "+model]=(loaded_model)  

                    print("SMOTE "+model)

SMOTE xgb_learning_rate_0.47_max_depth_3_n_estimators_183_subsample_0.72_colsample_bytree_0.83.joblib
SMOTE iforest_n_estimators_102_contamination_0_38_max_features_0_88.joblib
SMOTE rf_trees_n_estimators_101_max_depth_30.joblib
SMOTE MLP_hidden_layer_sizes_(50, 50)_activation_func_relu.joblib
SMOTE decision_tree_min_samples_split_2_depth_None.joblib
SMOTE cnn_f107_d43_dropout_0_45.keras
SMOTE lr_C_94.9354_solver_lbfgs.joblib


In [116]:

print("\n7. Tuning SVM with loops...")
best_svm_acc = 0
best_svm_model = None
best_svm_params = {}

for C in [0.1, 1.0, 10.0]:
    for kernel in ['rbf', 'linear']:
        if(C==10.0 and kernel=='linear'):
            break
        model = SVC(
            C=C,
            kernel=kernel,
            probability=True,   # needed for predict_proba / ROC AUC
            random_state=42
        )
        #model.fit(X_train_resampled, y_train_resampled)

        filename = f"savedSMOTE_models/svm/svm_C{C}_kernel_{kernel}.joblib"
        #joblib.dump(model, filename)
        #print(f"Saved: {filename}")
        model =joblib.load(filename)
        val_preds = model.predict(X_val)
        val_acc = accuracy_score(y_val, val_preds)

        if val_acc > best_svm_acc:
            best_svm_acc = val_acc
            best_svm_model = model
            bestmodel_name = "SMOTE svm_C{C}_kernel_{kernel}.joblib"
            best_svm_params = {'C': C, 'kernel': kernel}

best_models[bestmodel_name] = best_svm_model
print(f"Best SVM Params: {best_svm_params} | Val Accuracy: {best_svm_acc:.4f}")



7. Tuning SVM with loops...
Best SVM Params: {'C': 1.0, 'kernel': 'rbf'} | Val Accuracy: 0.7831


In [117]:
X_train_cnn = X_train_resampled.reshape(X_train_resampled.shape[0], X_train_resampled.shape[1], 1)
X_val_cnn = X_val.reshape(X_val.shape[0], X_val.shape[1], 1)
X_test_cnn = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)


In [118]:
print("\nGenerating final predictions on Test Set...")
for name, model in best_models.items():
    print(name)
    if name.endswith('.keras') :
        prob = model.predict(X_test_cnn, verbose=0).flatten()
        pred = (prob > 0.5).astype(int)
    elif name == 'SMOTE iforest_n_estimators_102_contamination_0_38_max_features_0_88.joblib':
        raw = model.predict(X_test)
        pred = np.where(raw == -1, 1, 0)
        # Use decision_function as anomaly score (lower = more anomalous → negate for proba-like score)
        prob = -model.decision_function(X_test)
        # Normalise to [0,1] so roc_auc_score works
        prob = (prob - prob.min()) / (prob.max() - prob.min() + 1e-9)
    else:
        pred = model.predict(X_test)
        prob = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else np.zeros(len(y_test))

    predictions[name] = {'pred': pred, 'prob': prob}

print("\n--- LOOP TUNING COMPLETE ---")



Generating final predictions on Test Set...
decision_tree_min_samples_split_2_depth_30.joblib
MLP_hidden_layer_sizes_(50,)_activation_func_relu.joblib
cnn_f59_d44.keras
rf_trees_n_estimators_58_max_depth_10.joblib
SMOTE xgb_learning_rate_0.47_max_depth_3_n_estimators_183_subsample_0.72_colsample_bytree_0.83.joblib
SMOTE iforest_n_estimators_102_contamination_0_38_max_features_0_88.joblib
SMOTE rf_trees_n_estimators_101_max_depth_30.joblib
SMOTE MLP_hidden_layer_sizes_(50, 50)_activation_func_relu.joblib
SMOTE decision_tree_min_samples_split_2_depth_None.joblib
SMOTE cnn_f107_d43_dropout_0_45.keras
SMOTE lr_C_94.9354_solver_lbfgs.joblib
SMOTE svm_C{C}_kernel_{kernel}.joblib

--- LOOP TUNING COMPLETE ---


#Results

In [120]:

print("--- GENERATING FINAL RESULTS AND SAVING TO CSV ---")

results_list = []

for name in best_models.keys():
    print(name)
    pred = predictions[name]['pred']
    prob = predictions[name]['prob']

    acc = accuracy_score(y_test, pred)
    prec = precision_score(y_test, pred, zero_division=0)
    rec = recall_score(y_test, pred, zero_division=0)
    f1 = f1_score(y_test, pred, zero_division=0)

    try:
        auc_score = roc_auc_score(y_test, prob)
    except Exception:
        auc_score = "N/A"
    try:
        PRAUC = average_precision_score(y_test, prob)
    except Exception:
        PRAUC = "N/A"

    results_list.append({
        'Model': name,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1 Score': f1,
        'ROC AUC': auc_score,
        "PR AUC": PRAUC
    })

results_df = pd.DataFrame(results_list)

print("\n--- FINAL TEST SET EVALUATION ---")
display(results_df)

csv_filename = 'results.csv'
results_df.to_csv(csv_filename, index=False)
print(f"\n Results successfully saved to {csv_filename}!")

--- GENERATING FINAL RESULTS AND SAVING TO CSV ---
decision_tree_min_samples_split_2_depth_30.joblib
MLP_hidden_layer_sizes_(50,)_activation_func_relu.joblib
cnn_f59_d44.keras
rf_trees_n_estimators_58_max_depth_10.joblib
SMOTE xgb_learning_rate_0.47_max_depth_3_n_estimators_183_subsample_0.72_colsample_bytree_0.83.joblib
SMOTE iforest_n_estimators_102_contamination_0_38_max_features_0_88.joblib
SMOTE rf_trees_n_estimators_101_max_depth_30.joblib
SMOTE MLP_hidden_layer_sizes_(50, 50)_activation_func_relu.joblib
SMOTE decision_tree_min_samples_split_2_depth_None.joblib
SMOTE cnn_f107_d43_dropout_0_45.keras
SMOTE lr_C_94.9354_solver_lbfgs.joblib
SMOTE svm_C{C}_kernel_{kernel}.joblib

--- FINAL TEST SET EVALUATION ---


,Model,Accuracy,Precision,Recall,F1 Score,ROC AUC,PR AUC
0,decision_tree_min_samples_split_2_depth_30.joblib,0.808903,0.898497,0.853860,0.875610,0.747975,0.882306
1,"MLP_hidden_layer_sizes_(50,)_activation_func_r...",0.815425,0.892592,0.870420,0.881367,0.871046,0.965140
2,cnn_f59_d44.keras,0.793739,0.845008,0.903954,0.873487,0.835020,0.955305
3,rf_trees_n_estimators_58_max_depth_10.joblib,0.818197,0.886118,0.882633,0.884372,0.879944,0.967196
4,SMOTE xgb_learning_rate_0.47_max_depth_3_n_est...,0.889287,0.956866,0.900021,0.927573,0.958299,0.988713
5,SMOTE iforest_n_estimators_102_contamination_0...,0.751834,0.828992,0.862968,0.845639,0.626721,0.858004
6,SMOTE rf_trees_n_estimators_101_max_depth_30.j...,0.797489,0.921343,0.812254,0.863366,0.882271,0.968174
7,"SMOTE MLP_hidden_layer_sizes_(50, 50)_activati...",0.787543,0.880336,0.845167,0.862393,0.839862,0.950933
8,SMOTE decision_tree_min_samples_split_2_depth_...,0.790478,0.912902,0.811426,0.859178,0.762088,0.889293
9,SMOTE cnn_f107_d43_dropout_0_45.keras,0.763085,0.892243,0.795280,0.840976,0.835141,0.953433



 Results successfully saved to results.csv!
